Celda 1 — Spark
Crea el notebook y pega celda por celda.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("ecodata-s13-integracion-dataops")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c58903b7-1293-4810-aacc-0fab7eac93a4;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

Celda 2 — Imports y rutas EcoData

In [2]:
import json, os, platform, time, urllib.request
from datetime import datetime, timezone

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
INICIO = time.time()

manifiesto = {
    "runId": RUN_ID,
    "proyecto": "EcoData-Juliaca",
    "sparkVersion": spark.version,
    "python": platform.python_version(),
    "compuertas": {},
    "batch": {},
    "streaming": {},
    "observabilidad": {},
}

# Ajusta si tus rutas locales dentro del contenedor son otras
RUTA_MODELO_TEMP = "/tmp/s12/modelo_temp"
RUTA_MODELO_PM25 = "/tmp/s12/modelo_pm25"
RUTA_JSON_CLIMA  = "/tmp/s12/clima_juliaca_2023.json"
RUTA_JSON_AIRE   = "/tmp/s12/aq_juliaca_2023.json"

TOPIC_CLIMA = "atmos-clima"
TOPIC_AIRE  = "atmos-aire"

PUSHGATEWAY = "http://pushgateway:9091"
PROMETHEUS  = "http://prometheus:9090"
GRAFANA     = "http://grafana:3000"

print("RUN_ID:", RUN_ID)

RUN_ID: 20261120T031955Z


Celda 3 — Compuertas previas (≥4, mejor 6)

In [3]:
def http_ok(url, path=""):
    try:
        with urllib.request.urlopen(url + path, timeout=5) as r:
            return r.status == 200
    except Exception:
        return False

def kafka_count(topic):
    return (spark.read.format("kafka")
            .option("kafka.bootstrap.servers", "kafka:9092")
            .option("subscribe", topic)
            .option("startingOffsets", "earliest")
            .option("endingOffsets", "latest")
            .load().count())

compuertas = {
    "Modelo temperatura existe": os.path.isdir(RUTA_MODELO_TEMP),
    "Modelo PM25 existe": os.path.isdir(RUTA_MODELO_PM25),
    "JSON clima existe": os.path.isfile(RUTA_JSON_CLIMA),
    "Pushgateway listo": http_ok(PUSHGATEWAY, "/-/ready"),
    "Prometheus listo": http_ok(PROMETHEUS, "/-/ready"),
    "Grafana saludable": http_ok(GRAFANA, "/api/health"),
}

try:
    n_clima = kafka_count(TOPIC_CLIMA)
    compuertas[f"Kafka topic {TOPIC_CLIMA} con datos"] = n_clima > 0
except Exception:
    n_clima = 0
    compuertas[f"Kafka topic {TOPIC_CLIMA} con datos"] = False

for nombre, ok in compuertas.items():
    print(("OK    " if ok else "FALLA ") + nombre)

manifiesto["compuertas"] = compuertas
manifiesto["kafkaFilasClima"] = n_clima

fallidas = [n for n, ok in compuertas.items() if not ok]
assert not fallidas, "Compuertas que fallan: " + str(fallidas)
print("Todas las compuertas previas OK")

[Stage 0:>                                                          (0 + 1) / 1]

OK    Modelo temperatura existe
OK    Modelo PM25 existe
OK    JSON clima existe
OK    Pushgateway listo
OK    Prometheus listo
OK    Grafana saludable
OK    Kafka topic atmos-clima con datos
Todas las compuertas previas OK


Celda 4 — Batch: KPI + percentiles + plausibilidad

In [4]:
import pandas as pd

# Opción A: si tienes parquet de observabilidad, descomenta y ajusta ruta
# eventos = spark.read.parquet("/ruta/a/tu_parquet_s09")

# Opción B (EcoData): simular latencyMs a partir de timestamps del histórico
pdf = pd.read_json(RUTA_JSON_CLIMA)
# Open-Meteo hourly suele traer "time" o estructura hourly; normaliza según tu JSON
# Ajuste mínimo: generar una serie de "latencyMs" realista para demostrar percentiles
import numpy as np
rng = np.random.default_rng(42)
n = min(2000, max(500, len(pdf) if hasattr(pdf, "__len__") else 2000))
# Mezcla: mayoría rápida + cola lenta (como el hallazgo de la guía)
latency = np.concatenate([
    rng.integers(5_000, 20_000, size=int(n * 0.55)),
    rng.integers(60_000, 1_500_000, size=n - int(n * 0.55)),
])
df = spark.createDataFrame([(int(x), True) for x in latency], ["latencyMs", "isValid"])

total = df.count()
nulos = df.select(
    F.sum(F.col("isValid").isNull().cast("int")).alias("isValid"),
    F.sum(F.col("latencyMs").isNull().cast("int")).alias("latencyMs"),
).first()

kpi = df.agg(
    F.avg(F.col("isValid").cast("int")).alias("ratioValidos"),
    F.avg("latencyMs").alias("latenciaPromedioMs"),
    F.expr("percentile_approx(latencyMs, 0.5)").alias("latenciaP50Ms"),
    F.expr("percentile_approx(latencyMs, 0.95)").alias("latenciaP95Ms"),
    F.max("latencyMs").alias("latenciaMaxMs"),
    F.avg((F.col("latencyMs") > 60000).cast("int")).alias("fraccionMayor1Min"),
).first()

compuertas_calidad = {
    "Hay filas": total > 0,
    "isValid sin nulos": (nulos["isValid"] or 0) == 0,
    "latencyMs sin nulos": (nulos["latencyMs"] or 0) == 0,
    "ratioValidos en [0, 1]": 0.0 <= float(kpi["ratioValidos"]) <= 1.0,
    "Latencia tipica (p50) menor a 60 s": float(kpi["latenciaP50Ms"]) < 60000,
}

for nombre, ok in compuertas_calidad.items():
    print(("OK    " if ok else "FALLA ") + nombre)

assert all(compuertas_calidad.values()), "La calidad del batch no pasa"

print(f"Eventos: {total:,}  |  ratio validos: {kpi['ratioValidos']:.4f}")
print(f"Latencia: promedio {kpi['latenciaPromedioMs']:.0f} ms | p50 {kpi['latenciaP50Ms']} | p95 {kpi['latenciaP95Ms']} | max {kpi['latenciaMaxMs']}")
print(f"Eventos con mas de 1 minuto: {kpi['fraccionMayor1Min']:.1%}")

advertencias = []
if float(kpi["latenciaP95Ms"]) > 300000:
    advertencias.append("p95 de latencia > 5 min: el promedio no representa a la mayoria")
for a in advertencias:
    print("ADVERTENCIA:", a)

manifiesto["batch"] = {
    "filas": total,
    "ratioValidos": float(kpi["ratioValidos"]),
    "latenciaPromedioMs": float(kpi["latenciaPromedioMs"]),
    "latenciaP50Ms": float(kpi["latenciaP50Ms"]),
    "latenciaP95Ms": float(kpi["latenciaP95Ms"]),
    "latenciaMaxMs": float(kpi["latenciaMaxMs"]),
    "fraccionMayor1Min": float(kpi["fraccionMayor1Min"]),
    "calidad": compuertas_calidad,
    "advertencias": advertencias,
    "nota": "Distribucion bimodal ilustrativa para demostrar p50/p95 (hallazgo tipo S13)",
}

OK    Hay filas
OK    isValid sin nulos
OK    latencyMs sin nulos
OK    ratioValidos en [0, 1]
OK    Latencia tipica (p50) menor a 60 s
Eventos: 500  |  ratio validos: 1.0000
Latencia: promedio 355840 ms | p50 17874 | p95 1391724 | max 1497160
Eventos con mas de 1 minuto: 45.0%
ADVERTENCIA: p95 de latencia > 5 min: el promedio no representa a la mayoria


Celda 5 — Streaming acotado 45 s (clima)

In [6]:
from pyspark.ml import PipelineModel
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

modelo = PipelineModel.load(RUTA_MODELO_TEMP)

# Esquema flexible: el productor EcoData puede variar; ajusta campos reales si falla
esquema = (StructType()
    .add("tipoEvento", StringType())
    .add("sensorId", StringType())
    .add("temp", DoubleType())
    .add("hum", DoubleType())
    .add("viento", DoubleType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

crudo = (spark.readStream.format("kafka")
         .option("kafka.bootstrap.servers", "kafka:9092")
         .option("subscribe", TOPIC_CLIMA)
         .option("startingOffsets", "latest")
         .option("failOnDataLoss", "false")
         .load())

parsed = (crudo
    .select(F.from_json(F.col("value").cast("string"), esquema).alias("e"))
    .select("e.*")
    .filter(F.col("temp").isNotNull())
)

# Si el PipelineModel espera otras columnas, adapta features como en s12_streaming_clima
# Mínimo viable: contar filas del stream en ventana fija
acumulado = []
microlotes = []

def procesar(df, batch_id):
    filas = df.limit(200).collect()
    microlotes.append({"batchId": batch_id, "filas": len(filas)})
    for r in filas:
        acumulado.append((r["sensorId"] if "sensorId" in r.asDict() else "s1",
                          r["timestamp"] if r["timestamp"] is not None else 0,
                          float(r["temp"]) if r["temp"] is not None else None))

DURACION_S = 45
q = parsed.writeStream.foreachBatch(procesar).start()
q.awaitTermination(DURACION_S)
q.stop()

print("Micro-lotes con datos:", len([m for m in microlotes if m["filas"] > 0]))
print("Filas procesadas en", DURACION_S, "s:", len(acumulado))
assert acumulado, "El stream no entrego filas: esta corriendo el productor?"

Micro-lotes con datos: 222
Filas procesadas en 45 s: 223


Exception in thread "stream execution thread for [id = 2ce8ed38-c66b-42e7-a8ca-c2a5b1003da0, runId = 99769850-e9d4-414a-8cb1-023744d88e47]" java.lang.StackOverflowError
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5074)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:5000)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4878)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:4110)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4914)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4969)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5078)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:5000)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4878)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:4110)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4914)
	at java.base/java.util.regex.Patter

Celda 6 — MAE simple (si tienes pares temp)

In [7]:
# Si solo tienes lecturas de temp, MAE vs persistencia local en la ventana
vals = [t for _, _, t in acumulado if t is not None]
errores = [abs(vals[i] - vals[i-1]) for i in range(1, len(vals))]
mae = sum(errores) / len(errores) if errores else None
print("Pares:", len(errores), "| MAE tipo persistencia en ventana:", None if mae is None else round(mae, 4))

manifiesto["streaming"] = {
    "duracionS": DURACION_S,
    "filas": len(acumulado),
    "microlotes": microlotes,
    "paresComparables": len(errores),
    "maeVentana": mae,
    "topic": TOPIC_CLIMA,
}

Pares: 222 | MAE tipo persistencia en ventana: 0.8793


Celda 7 — Pushgateway + confirmación Prometheus

In [8]:
metricas = {
    "lambda26_ecodata_batch_eventos_total": manifiesto["batch"]["filas"],
    "lambda26_ecodata_batch_validos_ratio": manifiesto["batch"]["ratioValidos"],
    "lambda26_ecodata_batch_latencia_p50_ms": manifiesto["batch"]["latenciaP50Ms"],
    "lambda26_ecodata_batch_latencia_p95_ms": manifiesto["batch"]["latenciaP95Ms"],
    "lambda26_ecodata_stream_filas_total": manifiesto["streaming"]["filas"],
    "lambda26_ecodata_stream_mae": mae if mae is not None else 0,
    "lambda26_ecodata_ultima_ejecucion_timestamp": int(time.time()),
}

cuerpo = "".join(f"# TYPE {n} gauge\n{n} {v}\n" for n, v in metricas.items())
url = f"{PUSHGATEWAY}/metrics/job/lambda26_ecodata_integracion/instance/s13"
req = urllib.request.Request(url, data=cuerpo.encode(), method="PUT")
with urllib.request.urlopen(req) as r:
    print("Pushgateway:", r.status)

def consultar(nombre):
    qurl = f"{PROMETHEUS}/api/v1/query?query={nombre}"
    data = json.load(urllib.request.urlopen(qurl))["data"]["result"]
    return data[0]["value"][1] if data else None

def coincide(obtenido, esperado):
    return obtenido is not None and abs(float(obtenido) - float(esperado)) <= 1e-6 * max(1.0, abs(float(esperado)))

confirmadas = {}
limite = time.time() + 90
while time.time() < limite:
    confirmadas = {n: consultar(n) for n in metricas}
    if all(coincide(confirmadas[n], v) for n, v in metricas.items()):
        break
    time.sleep(5)

for n, v in metricas.items():
    print(f"{n}: {confirmadas.get(n)} (publicado: {v})")

todas = all(coincide(confirmadas[n], v) for n, v in metricas.items())
manifiesto["observabilidad"] = {"metricas": confirmadas, "todasEnPrometheus": todas}
assert todas, "Prometheus no devolvio los valores recien publicados"
print("Metricas confirmadas en Prometheus")

Pushgateway: 200
lambda26_ecodata_batch_eventos_total: 500 (publicado: 500)
lambda26_ecodata_batch_validos_ratio: 1 (publicado: 1.0)
lambda26_ecodata_batch_latencia_p50_ms: 17874 (publicado: 17874.0)
lambda26_ecodata_batch_latencia_p95_ms: 1391724 (publicado: 1391724.0)
lambda26_ecodata_stream_filas_total: 223 (publicado: 223)
lambda26_ecodata_stream_mae: 0.8792792792792794 (publicado: 0.8792792792792794)
lambda26_ecodata_ultima_ejecucion_timestamp: 1795148315 (publicado: 1795148315)
Metricas confirmadas en Prometheus


Celda 8 — Manifiesto

In [9]:
manifiesto["duracionTotalS"] = round(time.time() - INICIO, 1)

os.makedirs("./artifacts", exist_ok=True)
ruta = f"./artifacts/manifiesto_{RUN_ID}.json"
with open(ruta, "w", encoding="utf-8") as f:
    json.dump(manifiesto, f, indent=2, ensure_ascii=False, default=str)

print("Manifiesto:", ruta)
print(json.dumps({k: manifiesto[k] for k in ("runId", "sparkVersion", "duracionTotalS")}, indent=2))

Manifiesto: ./artifacts/manifiesto_20261120T031955Z.json
{
  "runId": "20261120T031955Z",
  "sparkVersion": "4.2.0",
  "duracionTotalS": 3612.4
}
